In [0]:
import pandas as pd, os
FPM = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume/final_product_mapping/"
WS = os.path.dirname("/Workspace" + dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()) + "/"
print(sorted(os.listdir(FPM))); print(sorted(f for f in os.listdir(WS) if f.endswith(".csv")))
rx = pd.read_csv(FPM + "Kerendia_Rx_Final_Product_Mapping_2026_11_04.csv", dtype=str, keep_default_na=False)
print(rx.shape, rx.ACTIVE_JOURNEY.value_counts().to_dict())
pat = "OZEMPIC|MOUNJARO|TRULICITY|RYBELSUS|VICTOZA|LIRAGLUTIDE|BYDUREON|BYETTA|EXENATIDE|WEGOVY|ZEPBOUND|SAXENDA|KERENDIA|DAPAGLIFLOZIN|ADLYXIN|XULTOPHY|SOLIQUA|J0606"
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 70)
print(rx[rx.MATCH_VALUE.str.contains(pat)][["MATCH_TYPE","MATCH_VALUE","PRODUCT_GROUP","TREATMENT_CATEGORY","BRAND_LABEL","STRENGTH","CONDITION","ACTIVE_JOURNEY","patients_total","NOTE"]].to_string())
print(sorted(rx.TREATMENT_CATEGORY.unique())); print(sorted(rx.BRAND_LABEL.unique())); print(sorted(rx.CONDITION.unique()))
px = pd.read_csv(FPM + "Kerendia_Px_Final_Product_Mapping_2026_11_04.csv", dtype=str, keep_default_na=False)
print(px.shape); print(px[px.MATCH_VALUE.str.contains("J0606")].T)

In [0]:
pairs = {"Rx": ("Kerendia_Rx_Final_Product_Mapping_2026_11_04.csv", "Kerendia_Rx_Final.csv"),
         "Px": ("Kerendia_Px_Final_Product_Mapping_2026_11_04.csv", "Kerendia_Px_Final.csv"),
         "Dx": ("Kerendia_Dx_Final_Product_Mapping_2026_11_04.csv", "Kerendia_Dx_Final.csv"),
         "Lab": ("Kerendia_Lab_Final_Product_Mapping_2026_11_04.csv", "Kerendia_Lab_Final.csv"),
         "DxAdd": ("Kerendia_Dx_Flag_Additions_2026_11_04.csv", "Kerendia_Dx_Flag_Additions.csv")}
for k, (v, w) in pairs.items():
    a = pd.read_csv(FPM + v, dtype=str, keep_default_na=False)
    b = pd.read_csv(WS + w, dtype=str, keep_default_na=False)
    common = [c for c in b.columns if c in a.columns]
    ci = {c.upper(): c for c in a.columns}
    extra_v = [c for c in a.columns if c not in b.columns]; extra_w = [c for c in b.columns if c not in a.columns]
    # compare columns by case-insensitive name
    cols_ci = [c for c in b.columns if c.upper() in ci]
    a2 = a[[ci[c.upper()] for c in cols_ci]].copy(); a2.columns = cols_ci
    b2 = b[cols_ci].copy()
    num = [c for c in cols_ci if c.upper() == "PATIENTS_TOTAL"]
    for c in num:
        a2[c] = pd.to_numeric(a2[c], errors="coerce"); b2[c] = pd.to_numeric(b2[c], errors="coerce")
    same_rows = len(a2) == len(b2)
    eq = same_rows and a2.reset_index(drop=True).fillna(-1).equals(b2.reset_index(drop=True).fillna(-1))
    print(f"{k:6s} volume {a.shape} ws {b.shape} | equal on {len(cols_ci)} shared cols: {eq}")
    print(f"        only in volume: {extra_v}\n        only in workspace: {extra_w}")
    if same_rows and not eq:
        d = (a2.reset_index(drop=True).fillna(-1) != b2.reset_index(drop=True).fillna(-1))
        print("        differing cols:", d.sum()[d.sum() > 0].to_dict())

In [0]:
for k, cols in [("Dx", ["CKD_STAGE"]), ("Lab", ["PRIORITY", "VALID_MIN", "VALID_MAX"])]:
    v, w = pairs[k]
    a = pd.read_csv(FPM + v, dtype=str, keep_default_na=False); b = pd.read_csv(WS + w, dtype=str, keep_default_na=False)
    for c in cols:
        d = pd.DataFrame({"volume": a[c], "workspace": b[c]})[a[c] != b[c]]
        print(k, c, d.drop_duplicates().to_dict("records"))
        # same number?
        print("   numerically equal:", (pd.to_numeric(d.volume) == pd.to_numeric(d.workspace)).all())

In [0]:
import json, re, glob
ROOT = os.path.dirname(WS.rstrip("/")) + "/"
nbs = [p for p in glob.glob(ROOT + "**/*", recursive=True)
       if re.search(r"(0[1-6]_patient_journey|0[1-6]_claim|04_claims|00_cohort_config)", os.path.basename(p))
       and "Clone" not in p and "Archive" not in p and os.path.isfile(p)]
for p in sorted(nbs):
    try:
        src = open(p).read()
    except Exception as e:
        print("skip", p, e); continue
    if p.endswith(".ipynb"):
        src = "\n".join("".join(c["source"]) for c in json.loads(src)["cells"])
    lines = src.split("\n")
    hits = [i for i, l in enumerate(lines) if re.search(r"_Final\.csv|Flag_Additions|BASKET|PATIENTS_TOTAL|read_csv", l)]
    if hits:
        print("=====", p.replace(ROOT, ""))
        for i in hits:
            print(f"  {i:4d}: {lines[i].strip()[:200]}")

In [0]:
import hashlib
src = json.load(open(ROOT + "Patient_Journey_Validation/Rx Files/03_patient_journey_rx.ipynb"))
txt = "\n".join("".join(c["source"]) for c in src["cells"])
print([l for l in txt.split("\n") if "RX_BASKET_FOLDER = " in l or ("develop_hcp_claims" in l)][:4])
src5 = json.load(open(ROOT + "Patient_Journey_Validation/Px Files/05_patient_journey_px.ipynb"))
txt5 = "\n".join("".join(c["source"]) for c in src5["cells"])
print([l for l in txt5.split("\n") if "develop_hcp_claims" in l or "_BASKET_DIR = " in l][:4])
OTHER = "/Workspace/Users/patricio.guledjian.ext@bayer.com/kerendia_nba_3_1_develop_hcp_claims/NBA 3.1 - Dev/03_Feature_Engineering/03_HCP_level_features/Final Product Basket/"
print(os.path.exists(OTHER), sorted(os.listdir(OTHER)) if os.path.exists(OTHER) else None)
h = lambda p: hashlib.md5(open(p, "rb").read()).hexdigest() if os.path.exists(p) else None
for f in sorted(f for f in os.listdir(WS) if f.endswith(".csv")):
    print(f"{f:45s} same bytes: {h(WS + f) == h(OTHER + f)}")

In [0]:
def show_cells(path, pattern, maxlen=4000):
    s = json.load(open(ROOT + path))
    for c in s["cells"]:
        t = "".join(c["source"])
        if re.search(pattern, t) and not t.lstrip().startswith("%md"):
            print("-----", path.split("/")[-1]); print(t[:maxlen])
show_cells("Patient_Journey_Validation/Rx Files/03_patient_journey_rx.ipynb", r"def read_rx_basket")
show_cells("Patient_Journey_Validation/Px Files/05_patient_journey_px.ipynb", r"px_basket_raw = pd.read_csv")
show_cells("Patient_Journey_Validation/Combine Files/06_patient_journey_combine_3_1.ipynb", r"TREATMENT_CATEGORY|SGLT2|FLAG", 2500)

In [0]:
s5 = ["".join(c["source"]) for c in src5["cells"] if "px_basket_raw = pd.read_csv" in "".join(c["source"])][0]
print(s5[s5.find("_yn = lambda"):][:2500])
print("=" * 80)
# 3.0 typed GLP-1 / DPP-IV lists in PJ03
for c in src["cells"]:
    t = "".join(c["source"])
    for m in re.finditer(r"(GLP1?_?\w*|DPP\w*)\s*=\s*\[[^\]]+\]", t):
        print(m.group(0)[:1500]); print()
print("=" * 80)
cfg = json.load(open(ROOT + "00_cohort_config.ipynb"))
ct = "\n".join("".join(c["source"]) for c in cfg["cells"])
for m in re.finditer(r"HCP_BRANDS\s*=\s*[\[\(][^\]\)]+[\]\)]", ct):
    print(m.group(0))
for m in re.finditer(r".*(TREATMENT_CATEGOR|HCP_TREAT)\w*\s*=.*", ct):
    print(m.group(0)[:300])

In [0]:
# 1. Product mapping update 2026-11-05 (NBA 3.1, decision 2026-11-04).
# Source: the notebook-format baskets in this folder (same content as Sehar's 2026_11_04 volume files, checked above;
# the volume files differ only in number format and extra usage columns, which the notebook basket checks reject).
# Output: one dated file per type in the 3.1 volume final_product_mapping/. Sehar's 2026_11_04 files are not touched.
import hashlib
NEW_DATE = "2026_11_05"
CHANGE = f"3.1 change {NEW_DATE[:4]}-{NEW_DATE[5:7]}-{NEW_DATE[8:]}"

GLP1 = ["OZEMPIC", "MOUNJARO", "TRULICITY", "RYBELSUS", "VICTOZA", "LIRAGLUTIDE", "BYDUREON BCISE", "BYETTA",
        "EXENATIDE", "WEGOVY", "WEGOVY HD", "ZEPBOUND", "SAXENDA"]
RX_CHANGES = {n: ("GLP-1", "GLP-1", "GLP-1 / GIP-GLP-1 receptor agonist (Rx summary section 5, D3)") for n in GLP1}
RX_CHANGES["KERENDIA"] = ("KERENDIA", "KERENDIA", "Kerendia (finerenone): own category, not MRA, so it is not counted with the competitor MRA drugs (Rx summary section 5, D3)")
for n in ["DAPAGLIFLOZIN", "DAPAGLIFLOZIN/METFORMIN H"]:
    RX_CHANGES[n] = ("SGLT2", "", "Generic dapagliflozin: SGLT2 like Farxiga / Xigduo XR, brand label left empty (shown as SGLT2), Farxiga brand stays brand only (Rx summary section 5, D3)")
PX_CHANGES = {"J0606": ("DIALYSIS_DRUG", "Etelcalcetide: billed only for hemodialysis patients, same as the other dialysis drugs (was 'Phase B candidate')")}

md5 = lambda p: hashlib.md5(open(p, "rb").read()).hexdigest()
old_vol = {f: md5(FPM + f) for f in os.listdir(FPM)}   # to prove the 2026_11_04 files are untouched

def add_note(old, new):
    return f"{old}; {CHANGE}: {new}" if old else f"{CHANGE}: {new}"

# Rx
rx_ws = pd.read_csv(WS + "Kerendia_Rx_Final.csv", dtype=str, keep_default_na=False)
rx_new = rx_ws.copy()
for name, (cat, label, why) in RX_CHANGES.items():
    i = rx_new.index[(rx_new.MATCH_TYPE == "NAME") & (rx_new.MATCH_VALUE == name) & (rx_new.STRENGTH == "")]
    assert len(i) == 1, f"Rx: expected one row for {name}, found {len(i)}"
    assert rx_new.loc[i[0], "ACTIVE_JOURNEY"] == "N" and rx_new.loc[i[0], "TREATMENT_CATEGORY"] == "", f"Rx: {name} is not an inactive row"
    rx_new.loc[i[0], ["ACTIVE_JOURNEY", "TREATMENT_CATEGORY", "BRAND_LABEL"]] = ["Y", cat, label]
    rx_new.loc[i[0], "NOTE"] = add_note(rx_new.loc[i[0], "NOTE"], f"N -> Y, {cat}. {why}")

# Px
px_ws = pd.read_csv(WS + "Kerendia_Px_Final.csv", dtype=str, keep_default_na=False)
px_new = px_ws.copy()
for code, (grp, why) in PX_CHANGES.items():
    i = px_new.index[(px_new.MATCH_TYPE == "CODE") & (px_new.MATCH_VALUE == code)]
    assert len(i) == 1 and px_new.loc[i[0], "ACTIVE_DIALYSIS"] == "N", f"Px: {code} not found as one inactive row"
    px_new.loc[i[0], ["ACTIVE_DIALYSIS", "PX_GROUP"]] = ["Y", grp]
    px_new.loc[i[0], "NOTE"] = add_note("", f"N -> Y, {grp}. {why}")

# Unchanged: Dx, Lab, Dx flag additions (same notebook format)
unchanged = {"Dx": "Kerendia_Dx_Final.csv", "Lab": "Kerendia_Lab_Final.csv", "Dx_Flag_Additions": "Kerendia_Dx_Flag_Additions.csv"}
out = {f"Kerendia_Rx_Final_Product_Mapping_{NEW_DATE}.csv": rx_new,
       f"Kerendia_Px_Final_Product_Mapping_{NEW_DATE}.csv": px_new}
for k, f in unchanged.items():
    name = f"Kerendia_{k}_Flag_Additions_{NEW_DATE}.csv" if k == "Dx_Flag_Additions" else f"Kerendia_{k}_Final_Product_Mapping_{NEW_DATE}.csv"
    if k == "Dx_Flag_Additions":
        name = f"Kerendia_Dx_Flag_Additions_{NEW_DATE}.csv"
    out[name] = pd.read_csv(WS + f, dtype=str, keep_default_na=False)

for name, df in out.items():
    p = FPM + name
    assert not name.endswith("2026_11_04.csv"), "never overwrite Sehar's files"
    df.to_csv(p, index=False)
    print(f"written {name}: {df.shape}")

assert all(md5(FPM + f) == h for f, h in old_vol.items()), "a 2026_11_04 file changed"
print("2026_11_04 files unchanged:", sorted(old_vol))

In [0]:
# 2. Verification, independent of cell 1: re-read the written files and compare with Sehar's 2026_11_04 volume files
#    cell by cell (numbers compared as numbers, column names case-insensitive). Expected: Rx 16 rows, Px 1 row, others 0.
def norm(df):
    d = df.copy(); d.columns = [c.upper() for c in d.columns]
    for c in d.columns:
        num = pd.to_numeric(d[c], errors="coerce")
        d[c] = d[c].where(num.isna() | (d[c] == ""), num.map(lambda x: f"{x:g}"))
    return d
kinds = [("Rx", "Kerendia_Rx_Final_Product_Mapping"), ("Px", "Kerendia_Px_Final_Product_Mapping"),
         ("Dx", "Kerendia_Dx_Final_Product_Mapping"), ("Lab", "Kerendia_Lab_Final_Product_Mapping"),
         ("DxAdd", "Kerendia_Dx_Flag_Additions")]
changed = {}
for k, stem in kinds:
    new = norm(pd.read_csv(f"{FPM}{stem}_2026_11_05.csv", dtype=str, keep_default_na=False))
    old = norm(pd.read_csv(f"{FPM}{stem}_2026_11_04.csv", dtype=str, keep_default_na=False))
    assert len(new) == len(old) and (new.MATCH_VALUE.values == old.MATCH_VALUE.values).all(), f"{k}: rows differ"
    cols = [c for c in new.columns if c in old.columns]
    diff = new[cols].ne(old[cols])
    rows = diff.any(axis=1)
    changed[k] = new.loc[rows, "MATCH_VALUE"].tolist()
    print(f"{k:6s} rows {len(new)} | rows changed: {rows.sum()} | columns changed: {sorted(diff.columns[diff.any()])} | "
          f"columns dropped vs volume file: {sorted(set(old.columns) - set(new.columns))}")
assert sorted(changed["Rx"]) == sorted(RX_CHANGES) and changed["Px"] == ["J0606"], changed
assert not changed["Dx"] and not changed["Lab"] and not changed["DxAdd"]

rx5 = pd.read_csv(f"{FPM}Kerendia_Rx_Final_Product_Mapping_2026_11_05.csv", dtype=str, keep_default_na=False)
display(rx5[rx5.MATCH_VALUE.isin(RX_CHANGES)][["MATCH_VALUE", "TREATMENT_CATEGORY", "BRAND_LABEL", "ACTIVE_JOURNEY", "PATIENTS_TOTAL", "NOTE"]])
print(rx5.ACTIVE_JOURNEY.value_counts().to_dict(), "| active categories:",
      rx5[rx5.ACTIVE_JOURNEY == "Y"].TREATMENT_CATEGORY.value_counts().to_dict())

# PJ03 basket check (function copied from the notebook source, run on the new file)
exec(["".join(c["source"]) for c in src["cells"] if "def read_rx_basket" in "".join(c["source"])][0].split("rx_basket_base, rx_basket_overrides")[0])
b_base, b_over, b_all = read_rx_basket(f"{FPM}Kerendia_Rx_Final_Product_Mapping_2026_11_05.csv")
print(f"PJ03 check OK: {len(b_base)} active product names (2026_11_04: {len(read_rx_basket(WS + 'Kerendia_Rx_Final.csv')[0])}), {len(b_over)} strength labels")

# PJ05 basket check (cell copied from the notebook source up to the display, run on the new file)
_c5 = s5.split("_active = px_basket")[0]
_c5 = re.sub(r'PX_BASKET_PATH = .*', f'PX_BASKET_PATH = "{FPM}Kerendia_Px_Final_Product_Mapping_2026_11_05.csv"', _c5)
_c5 = re.sub(r'PX_EVENT_LIST_PATH = .*', f'PX_EVENT_LIST_PATH = "{WS}Kerendia NBA 3.1 - PX Event List.csv"', _c5)
exec(_c5)
print("PJ05 check OK; ACTIVE_DIALYSIS = Y codes:", (px_basket.ACTIVE_DIALYSIS == "Y").sum(), "| J0606:",
      px_basket.loc[px_basket.PROCEDURE_CODE == "J0606", ["PX_GROUP", "ACTIVE_DIALYSIS"]].values.tolist())

In [0]:
pj = {"PJ01": "Patient_Journey_Validation/Lab Files/01_patient_journey_labs.ipynb",
      "PJ02": "Patient_Journey_Validation/Dx Files/02_patient_journey_dx.ipynb",
      "PJ03": "Patient_Journey_Validation/Rx Files/03_patient_journey_rx.ipynb",
      "PJ04": "Patient_Journey_Validation/T2D-T1D-HF Files/04_patient_journey_dm_hf_flags.ipynb",
      "PJ05": "Patient_Journey_Validation/Px Files/05_patient_journey_px.ipynb"}
for k, p in pj.items():
    cells = json.load(open(ROOT + p))["cells"]
    for i, c in enumerate(cells, 1):
        t = "".join(c["source"])
        if t.lstrip().startswith("%md"):
            continue
        for l in t.split("\n"):
            if re.search(r"(_BASKET_PATH|BASKET_DIR|BASKET_FOLDER|EVENT_LIST_PATH|dx_basket|dx_additions)\s*[=:]|\"dx_(basket|additions)\"", l):
                print(f"{k} cell {i}: {l.strip()[:180]}")

In [0]:
# 3. The loader in 00_cohort_config (cell "Product mapping files"), run from its source:
#    latest = 2026_11_05; a pinned date works; Sehar's 2026_11_04 files (other format) are rejected with a clear message.
_cfg_cells = ["".join(c["source"]) for c in json.load(open(ROOT + "00_cohort_config.ipynb"))["cells"]]
_loader = [t for t in _cfg_cells if "def product_mapping_path" in t][0]
_g = {"NBA31_VOLUME": FPM.split("final_product_mapping/")[0], "get_run_param": lambda n, d=None: d}
exec(_loader, _g)
for k in _g["PRODUCT_MAPPING_FILES"]:
    try:
        _g["product_mapping_path"](k, "2026_11_04"); print(f"{k}: 2026_11_04 accepted (this file was already in the notebook format)")
    except AssertionError as e:
        print(f"{k}: 2026_11_04 rejected -> {str(e).split(FPM)[-1][:150]}")
assert _g["product_mapping_path"]("rx", "2026_11_05").endswith("Kerendia_Rx_Final_Product_Mapping_2026_11_05.csv")
try:
    _g["product_mapping_path"]("rx", "2026_01_01")
except AssertionError as e:
    print("missing date ->", str(e)[:120])

In [0]:
# 4. Source check of PJ01-PJ05 after the switch (reads the saved notebooks):
#    each runs 00_cohort_config, reads its basket(s) through product_mapping_path, and has no old basket file / other-project path left.
expect = {"PJ01": ["lab"], "PJ02": ["dx"], "PJ03": ["rx"], "PJ04": ["dx", "dx_flag_additions"], "PJ05": ["px"]}
old = r"Kerendia_(Rx|Px|Dx|Lab)_Final\.csv|Kerendia_Dx_Flag_Additions\.csv|develop_hcp_claims"
for k, p in pj.items():
    cells = json.load(open(ROOT + p))["cells"]
    code = ["".join(c["source"]) for c in cells if c["cell_type"] == "code" and not "".join(c["source"]).lstrip().startswith("%md")]
    runs = [l.strip() for t in code for l in t.split("\n") if l.strip().startswith("%run")]
    used = sorted(set(re.findall(r'product_mapping_path\("(\w+)"\)', "\n".join(code))))
    left = [f"cell {i}: " + l.strip()[:120] for i, c in enumerate(cells, 1) if c["cell_type"] == "code"
            for l in "".join(c["source"]).split("\n") if re.search(old, l) and not l.strip().startswith("#")
            and not "".join(c["source"]).lstrip().startswith("%md")]
    print(f"{k}: %run {runs} | loader kinds {used} (expected {expect[k]}) | old paths left in code: {left or 'none'}")
    if not (used == sorted(expect[k]) and not left):
        print("   ^ NOT OK")
# PJ04 output path: where is it finally written?
_c4 = ["".join(c["source"]) for c in json.load(open(ROOT + pj["PJ04"]))["cells"]]
for i, t in enumerate(_c4, 1):
    for l in t.split("\n"):
        if "output_path" in l or "t2d_30_table" in l or ".save(" in l:
            print(f"PJ04 cell {i}: {l.strip()[:170]}")

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
%run "../00_cohort_config"

In [0]:
# 5. HCP side effects of the 2026-11-05 Rx mapping, measured on the Rx pull the HCP notebooks read today
#    (latest full run, built with the 2026_11_04 mapping). Read only.
import pandas as pd
GLP1 = ["OZEMPIC", "MOUNJARO", "TRULICITY", "RYBELSUS", "VICTOZA", "LIRAGLUTIDE", "BYDUREON BCISE", "BYETTA",
        "EXENATIDE", "WEGOVY", "WEGOVY HD", "ZEPBOUND", "SAXENDA"]
NEW = {**{n: ("GLP-1", "GLP-1") for n in GLP1}, "KERENDIA": ("KERENDIA", "KERENDIA"),
       "DAPAGLIFLOZIN": ("SGLT2", ""), "DAPAGLIFLOZIN/METFORMIN H": ("SGLT2", "")}
# The 3.1 run folder only exists after the next full run: use the latest rx_hcp_pull of the 3.0 query output (read only)
import os, glob
_roots = [HCP_QUERY_OUTPUT, NBA31_QUERY_OUTPUT, HCP_VOLUME + "notebook_output/", NBA31_PJ_OUTPUT]
_cands = sorted({p.rsplit("/_delta_log", 1)[0] for r in _roots
                 for pat in ["*/rx_hcp_pull/_delta_log", "*/_stage/rx_hcp_pull/_delta_log", "*/*/rx_hcp_pull/_delta_log"]
                 for p in glob.glob(r + pat)})
for c in _cands:
    print(f"{os.path.getmtime(c + '/_delta_log'):.0f}  {c}")
_full = [c for c in _cands if "validation" not in c]
_pull_path = builtins.max(_full or _cands, key=lambda c: os.path.getmtime(c + "/_delta_log"))
pull = spark.read.format("delta").load(_pull_path)
print(f"Rx pull: {_pull_path}\ncolumns: {pull.columns}")

# a) the 16 changed products (+ the 2 GLP-1 insulin combos already active) as they are in the pull today
cur = (pull.filter(F.col("PRODUCT_NAME").isin(list(NEW) + ["XULTOPHY 100/3.6", "SOLIQUA 100/33"]))
       .groupBy("PRODUCT_NAME", "TREATMENT_CATEGORY", "CATEGORY", "RX_SOURCE")
       .agg(F.count("*").alias("CLAIMS"), F.countDistinct(NPI_COL).alias("NPIS"))
       .orderBy(F.desc("CLAIMS"))).toPandas()
display(cur)

# b) treatment categories and brand values in the pull today (all claims) -> NB01 pivot values
cats = pull.groupBy("TREATMENT_CATEGORY", "RX_SOURCE").agg(F.count("*").alias("CLAIMS")).orderBy("TREATMENT_CATEGORY").toPandas()
display(cats)
brands = pull.filter(F.expr(HCP_BRAND_FILTER)).groupBy("CATEGORY").agg(F.count("*").alias("CLAIMS")).toPandas()
print("HCP_BRANDS:", HCP_BRANDS, "\nbrand values in pull:", brands.set_index("CATEGORY").CLAIMS.to_dict())

# c) NB01 column names: category pivot and brand pivot share the prefix TOTAL_RX_CLAIMS_ (cells 13-14)
_cat_now = {c.rstrip() for c in cats.TREATMENT_CATEGORY.dropna()}
_cat_new = _cat_now | {v[0] for v in NEW.values()}
_brand   = set(brands.CATEGORY.dropna()) | {"GLP-1"}
for label, cs in [("today", _cat_now), ("2026-11-05 mapping", _cat_new)]:
    clash = sorted({clean_col_name("TOTAL_RX_CLAIMS_" + c) for c in cs} & {clean_col_name("TOTAL_RX_CLAIMS_" + b) for b in _brand})
    print(f"Category vs brand column clash ({label}): {clash or 'none'}")

In [0]:
# 5b. What CATEGORY (brand) do journey SGLT2 / MRA rows get today: generic dapagliflozin (no BRAND_LABEL) will get the same
display(pull.filter(F.col("TREATMENT_CATEGORY").isin("SGLT2", "MRA"))
        .groupBy("TREATMENT_CATEGORY", "CATEGORY").agg(F.count("*").alias("CLAIMS"), F.countDistinct("PRODUCT_NAME").alias("PRODUCTS"))
        .orderBy("TREATMENT_CATEGORY", F.desc("CLAIMS")))
# NB01 pivot families where the category pivot and the brand pivot share a prefix (cells 13-17): a GLP-1 category
# value clashes with the GLP-1 brand value in each of them
families = {"base": ("category", "brand"), "rejected": ("rej_category", "rej_brand"), "duration": ("dur_category", "dur_brand"),
            "payer": ("payer_category", "payer_brand"), "age": ("age_category", "age_brand"), "gender": ("gender_category", "gender_brand")}
print("Each family x 4 windows (+ PCT_ / RUNNING_ variants) would get duplicate GLP_1 columns:", list(families))

In [0]:
# Scratch (for the fill-missing / lag note): sources of the window helpers in 00_cohort_config, and the bounds
import inspect
for fn in ["build_feature_group", "filter_to_window", "_build_bounds", "get_cohort_bounds"]:
    f = globals().get(fn)
    print(f"===== {fn}: {'MISSING' if f is None else inspect.signature(f)}")
    if f is not None:
        print(inspect.getsource(f)[:3000])
print("AVG_EXTRA_MONTH =", globals().get("AVG_EXTRA_MONTH"), "| OFFSET_DAYS =", OFFSET_DAYS)

In [0]:
# Scratch: toy test for the note. NPI A: 2 claims in Jan 2025, 1 in May 2025. NPI B: 1 claim in Feb 2025 only.
# Cohort 2025-05 at offset 0, windows 1 / 3 / 12 months. Compare 3.0 (monthly table, +/- grid, rowsBetween) with 3.1.
from pyspark.sql import Window
toy = spark.createDataFrame([("A", "2025-01-10", "X"), ("A", "2025-01-20", "X"), ("A", "2025-05-03", "X"),
                             ("B", "2025-02-14", "X")], "NPI string, SVC_DT string, CAT string").withColumn("SVC_DT", F.to_date("SVC_DT"))
monthly = toy.withColumn("M", F.trunc("SVC_DT", "MM")).groupBy("NPI", "M").pivot("CAT", ["X"]).count()
grid = (toy.select("NPI").distinct()
        .crossJoin(spark.sql("select explode(sequence(date'2024-06-01', date'2025-05-01', interval 1 month)) as M")))
def lag30(df, n):   # step3_lag: SUM over the last n ROWS
    return df.withColumn(f"L{n}", F.sum("X").over(Window.partitionBy("NPI").orderBy("M").rowsBetween(-n + 1, 0)))
for label, base in [("3.0 with grid", grid.join(monthly, ["NPI", "M"], "left").na.fill(0)), ("3.0 without grid", monthly)]:
    r = lag30(lag30(lag30(base, 1), 3), 12).filter(F.col("M") == "2025-05-01")
    print(label, {x.NPI: (x.L1, x.L3, x.L12) for x in r.collect()},
          "| B has a May row:", r.filter(F.col("NPI") == "B").count() == 1)
cb = get_cohort_bounds("2025-05", offset_days=0)
print("3.1 bounds (offset 0, 2025-05):", cb["window_bounds"])
new = build_feature_group(toy, "NPI", "SVC_DT", "CAT", "N_", {w: cb["window_bounds"][w] for w in [1, 3, 12]}, pivot_values=["X"])
print("3.1 build_feature_group:", {x.NPI: (x.N_X_IN_LAST_1_MONTH, x.N_X_IN_LAST_3_MONTH, x.N_X_IN_LAST_12_MONTH) for x in new.collect()})
print("offset 7, 2026-03, 1 month:", get_cohort_bounds("2026-03", offset_days=7)["window_bounds"][1])

In [0]:
# 6. Option C (2026-11-05): hcp_rx_categories (00_cohort_config) maps TREATMENT_CATEGORY GLP-1 -> 'Other- glucose mgmt'.
# a) known cases
_t = spark.createDataFrame([("GLP-1",), ("metformin ",), ("SGLT2",), ("Other- glucose mgmt",), (None,), ("KERENDIA",)], "TREATMENT_CATEGORY string")
_got = [r[0] for r in hcp_rx_categories(_t).collect()]
_exp = ["Other- glucose mgmt", "metformin", "SGLT2", "Other- glucose mgmt", None, "KERENDIA"]
print("known cases:", "PASS" if _got == _exp else f"FAIL {_got}")

# b) simulate what PJ03 writes with the 2026_11_05 mapping: the GLP-1 products get TREATMENT_CATEGORY 'GLP-1'
_sim = pull.withColumn("TREATMENT_CATEGORY",
                       F.when(F.col("PRODUCT_NAME").isin(GLP1), F.lit("GLP-1")).otherwise(F.col("TREATMENT_CATEGORY")))
def _cat_counts(df):
    return {r[0]: r[1] for r in df.groupBy("TREATMENT_CATEGORY").count().collect()}
# today's pull with the 3.0 cleanup only (trailing spaces), = what NB01 / NB04 counted in the 2026-11-04 full run
today   = _cat_counts(pull.withColumn("TREATMENT_CATEGORY", F.regexp_replace("TREATMENT_CATEGORY", r"[\s]*$", "")))
sim_raw = _cat_counts(_sim)
sim_c   = _cat_counts(hcp_rx_categories(_sim))
print("simulated pull without option C has GLP-1 category:", "GLP-1" in sim_raw, f"({sim_raw.get('GLP-1', 0):,} claims)")
print("with option C, category counts equal today's pull:", sim_c == today)
_brand_vals = {r[0] for r in hcp_rx_categories(_sim).filter(F.expr(HCP_BRAND_FILTER)).select("CATEGORY").distinct().collect()}
print("category vs brand clash with option C:", sorted(set(sim_c) & _brand_vals) or "none",
      "| without:", sorted(set(sim_raw) & _brand_vals) or "none")